In [11]:
import pandas as pd
import numpy as np
import warnings 
warnings.filterwarnings('ignore')

### 개요
- '공공자전거 대여소 정보(24.6월 기준).csv'에 LCD, QR에 있는 수치가 각 스테이션의 수용 한계치라고 가정.
- 주소를 이용하여 '은평구_스테이션_군집화_1차.csv'와 비교하여 같은 주소라면 LCD, QR의 값을 은평구_스테이션_군집화_1차.csv에 입력
- 입력된 파일을 '은평구_스테이션_군집화_1차_자전거댓수_추가.csv'로 저장함


In [ ]:
df = pd.read_csv('../../Data/은평구_스테이션_군집화_1차.csv')
df.drop('Unnamed: 0',axis=1,inplace=True)
df['주소1'] = df['주소1'].str.replace('서울 특별시','')
df['주소1'] = df['주소1'].str.replace('서울특별시','')
df['주소1'] = df['주소1'].str.replace('은평구','')
df['주소'] = df['주소1'].str.replace(' ','')
df1 = df[df['cluster_12_custom'] == 6]
df1.reset_index(drop=True, inplace=True)
df1['주소'] = df1['주소1'].str.replace(' ','')
df1

bycle = pd.read_csv('../../Data/Zero/Temp/공공자전거 대여소 정보(24.6월 기준).csv')
bycle.rename({'대여소\r\n번호':'대여소번호','설치\r\n시기':'설치시기','운영\r\n방식':'운영방식'},axis=1, inplace=True)
bycle = bycle[bycle.자치구 == '은평구']
bycle.fillna(0,inplace=True)
bycle['주소'] = bycle['상세주소'].str.replace(' ','')
bycle['주소'] = bycle['주소'].str.replace('서울특별시은평구','')
bycle['주소'] = bycle['주소'].str.replace('은평구','')

bycle

,대여소번호,보관소(대여소)명,자치구,상세주소,위도,경도,설치시기,LCD,QR,운영방식,주소
991,900,은평예술회관,은평구,서울특별시 은평구 녹번동 85-4 은평예술회관,37.603943,126.927696,2016-09-02,20.0,20.0,QR,녹번동85-4은평예술회관
992,902,진관동 은빛초등학교,은평구,서울특별시 은평구 진관동 45-21 진관동 은빛초등학교,37.643108,126.918221,2016-09-02,10.0,0.0,LCD,진관동45-21진관동은빛초등학교
993,903,은평뉴타운 아이파크,은평구,서울특별시 은평구 진관동 13-20 은평뉴타운 아이파크,37.645866,126.927391,2016-09-02,20.0,20.0,QR,진관동13-20은평뉴타운아이파크
994,904,은평뉴타운 푸르지오,은평구,서울특별시 은평구 진관동 5-8 은평뉴타운 푸르지오,37.648674,126.930702,2016-09-02,18.0,0.0,LCD,진관동5-8은평뉴타운푸르지오
995,905,구파발역 2번출구,은평구,서울특별시 은평구 진관동 86-31 구파발역 2번출구,37.636234,126.918999,2016-09-02,11.0,10.0,QR,진관동86-31구파발역2번출구
...,...,...,...,...,...,...,...,...,...,...,...
1084,4686,우물골 근린공원,은평구,은평구 진관2로 60 323동 앞,37.634068,126.925392,2022-08-26,0.0,7.0,QR,진관2로60323동앞
1085,4687,역촌센트레빌아파트 101동 앞,은평구,은평구 갈현로3나길 23 101동 앞,37.603542,126.906723,2022-08-26,0.0,10.0,QR,갈현로3나길23101동앞
1086,4689,은평서해그랑블아파트앞,은평구,은평구 서오릉로 253,37.617050,126.906586,2022-11-18,0.0,10.0,QR,서오릉로253
1087,4690,은평뉴타운폭포동 힐스테이트4단지,은평구,은평구 진관동 151-12,37.628342,126.933815,2023-01-04,0.0,5.0,QR,진관동151-12


In [ ]:
# bycle[bycle.자치구 == '은평구']
addr = df.주소.unique().tolist()
bycle.fillna(0,inplace=True)

for add in addr:
    lcd_match = bycle.loc[bycle['주소'].str.contains(add, na=False), 'LCD']
    qr_match = bycle.loc[bycle['주소'].str.contains(add, na=False), 'QR']

    if len(lcd_match) > 0:
        df.loc[df['주소'] == add, 'LCD'] = lcd_match.iloc[0]

    if len(qr_match) > 0:
        df.loc[df['주소'] == add, 'QR'] = qr_match.iloc[0]
df.drop('주소',axis=1, inplace=True) 
df   


,대여소_ID,주소1,주소2,위도,경도,1분기_전체건수,2분기_전체건수,3분기_전체건수,4분기_전체건수,연간_전체건수,cluster_12_custom,LCD,QR
0,ST-453,진관동 86-31,구파발역 2번출구,37.636234,126.918999,8392,19134,14804,10860,53190,1,11.0,10.0
1,ST-1483,진관2로 15-46,은평구 진관2로 15-46,37.639259,126.918907,3696,8120,6612,4986,23414,1,10.0,0.0
2,ST-1481,진관2로 지하 15-25,구파발역 환승센터,37.638252,126.919456,3112,6034,4838,4348,18332,1,15.0,0.0
3,ST-1329,진관3로 77,은평뉴타운구파발9단지,37.642609,126.921478,2696,5372,4268,3298,15634,1,10.0,0.0
4,ST-2244,진관4로 26 진관고등학교,NaN,37.643509,126.924103,2082,5474,4326,3502,15384,1,0.0,10.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
93,ST-462,신사동 352-2,NaN,37.590809,126.913689,10730,23348,15650,14046,63774,12,NaN,NaN
94,ST-463,증산동 199-8,증산역 4번출구,37.584381,126.909897,7233,12914,9790,8834,38771,12,10.0,0.0
95,ST-2257,가좌로 255,NaN,37.591812,126.914833,6633,13360,8368,7994,36355,12,0.0,10.0
96,ST-1034,증산동 239,디지털미디어 시티역 4번출구,37.577202,126.902351,3796,5736,4514,4626,18672,12,10.0,0.0


In [16]:
df.to_csv('../../Data/은평구_스테이션_군집화_1차_자전거댓수_추가.csv',index=None)